# EDA – Personalized E-Commerce Recommendation System

Run `python -m src.data_preprocessing --dataset demo` (or `real`) first. Set `DATASET` below. **If DATASET is `demo`, everything here is synthetic smoke-test data.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.utils.common import load_config
from src.data_preprocessing import load_processed, eda_statistics
sns.set_theme(style='whitegrid')
DATASET = 'demo'  # or 'real'
cfg = load_config()
tx, art, cust, meta = load_processed(DATASET, cfg)
print('synthetic:', meta['is_synthetic'])
tx.head()

## Data quality

In [ ]:
pd.Series(meta['quality_report']).to_frame('value')

In [ ]:
pd.DataFrame({'articles': art.isna().mean(), }).query('articles>0'), cust.isna().mean()[lambda s: s>0]

## Activity & popularity

In [ ]:
stats = eda_statistics(art, cust, tx)
pd.Series({k: v for k, v in stats.items() if not isinstance(v, dict)})

In [ ]:
per_cust = tx.groupby('customer_id').size()
per_cust.describe()

In [ ]:
j = tx.merge(art[['article_id','product_group_name','product_type_name']], on='article_id')
j['product_group_name'].value_counts().sort_values().plot.barh(figsize=(8,4), title='Transactions by product group');

## Seasonality and trend

In [ ]:
tx.set_index('t_dat').resample('W').size().plot(figsize=(10,3.5), title='Weekly transactions');

In [ ]:
top = j['product_type_name'].value_counts().head(8).index
m = j[j.product_type_name.isin(top)].assign(month=lambda d: d.t_dat.dt.month)
p = m.pivot_table(index='product_type_name', columns='month', values='article_id', aggfunc='size', fill_value=0)
sns.heatmap(p.div(p.sum(1), axis=0), cmap='mako_r'); plt.title('Monthly share of purchases by product type');

## Repeat purchases

In [ ]:
pair = tx.groupby(['customer_id','article_id']).size()
print('share of customer-item pairs bought more than once:', round((pair>1).mean(), 4))
pair.clip(upper=6).value_counts().sort_index()

## Interaction matrix sparsity

In [ ]:
from src.data_preprocessing import build_interaction_matrix
M, users, items = build_interaction_matrix(tx)
print(M.shape, 'density =', M.nnz / (M.shape[0]*M.shape[1]))